In [ ]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Custom MLP Model
- This is a simple implementation of a structure of a Multilayer percepreton neural network for study. Not intended to be use in any competitons ( pytorch and keras\tf are way better :D). 

In [ ]:
def addBias(X,n_hidden):
    bias = np.ones((1,X.shape[1]))
    X = np.concatenate([X, bias],axis =0)
    
    return X,(n_hidden+1)

def getLayers(X, y, n_hidden_layers):
    """Function to get the sizes of NN"""
    return (X.shape[0], y.shape[0], n_hidden_layers)

def initializeParameters(X, layers):
    """Function to initialize the parameters"""
    in_,out_,h = layers
    
    #wights
    W1 = np.random.random((h, in_-1))
    W2 = np.random.random(((out_), h-1))
    #bias
    b1 = np.ones((h,1))
    b2 = np.ones((1,out_))
    
    W1 = np.concatenate([W1, b1],axis =1)
    W2 = np.concatenate([W2,b2],axis = 1)
    
    return {'W1':W1, 
            'W2':W2}

def sigmoid(z):
    """Activation Function"""
    return (1/(1+np.exp(-z)))

def forward(X, params, layers):
    
    in_,out_,h = layers
    W1 = params['W1']
    W2 = params['W2']

    Z1 = np.dot(params['W1'], X) #+ params['b1']
    A1 = sigmoid(Z1)
    Z2 = np.dot(params['W2'], A1) #+ params['b2']
    y_hat = sigmoid(Z2)
    
    
    return {'Z1':Z1,
            'A1':A1,
            'Z2':Z2,
            'y_hat':y_hat}

def loss_func(y, feed):
    
    samples = y.shape[1]
    
    loss = np.sum((y- feed['y_hat'])**2)/samples
    
    return loss

def backpropagation(X, y, layers, params, feed):
    
    in_,out_,h = layers
    samples = y.shape[1]
    W1 = params['W1']
    W2 = params['W2']
    A1 = feed['A1']
    y_hat = feed['y_hat']
    
    dZ2 = 2*(y_hat - y) # loss
    dW2 = (1/samples) * np.dot(dZ2, A1.T)
    dZ1 = np.dot(W2.T, dZ2) * (1 - A1**2)
    dW1 = (1/samples) * np.dot(dZ1, X.T)
    
    return {'grad1':dW1,
            'grad2':dW2}


def update_params(grads, params, lr):
    
    W1 = params['W1']
    W2 = params['W2']
    
    grad1 = grads['grad1']
    grad2 = grads['grad2']
    
    W1 -= lr*(grad1)
    W2 -= lr*(grad2)

    return {'W1':W1, 
            'W2':W2}

def accuracy(y,  y_hat):
    return (np.sum(np.equal(y,np.vectorize(int)(y_hat >0.5))) /  (y.shape[1])) * 100

def train_model(X, X_test, y, y_test,epochs, n_hidden,lr, lr_factor =0.9 ):
    
    loss_l = []
    acc_l = []
    X,n_hidden = addBias(X,n_hidden)
    layers = getLayers(X,y,n_hidden)
    
    params = initializeParameters(X, layers)
    loss = 1
    acc = 0
    acc_val = 0
    best_acc  = 0
    y_t = []
    acc_val_l = []
    loss_val_l = []
    loss_val = 1
    
    
    for i in range(epochs):
        if (i+1)%1000 == 0:
            print(f'Epoch{i+1}/{epochs} - Loss: {np.round(loss,4)} - Acc: {np.round(acc,4)}% - lr: {lr} - Loss Val: {loss_val} - Acc Val: {acc_val}')
        if (i+1)%1000 == 0:
            lr *=lr_factor
        
        feed = forward(X, params, layers)
        loss = (loss_func(y, feed))
        acc  = accuracy(y,feed['y_hat'])
        loss_l.append(loss)
        acc_l.append(acc)
        
      
        grads = backpropagation(X, y, layers, params, feed)
        params = update_params(grads, params, lr=lr)
        
        
        if acc > best_acc:
            best_model = copy.deepcopy(params)
            best_acc = acc
          
        loss_val, acc_val = validate(X_test, y_test,params, layers)
        acc_val_l.append(acc_val)
        loss_val_l.append(loss_val)
        
    history = {'loss':loss_l, 'acc':acc_l, 'acc_val':acc_val_l,'loss_vall':loss_val_l,}
    print(f'Best Accuracy:{best_acc}')
    return {'history': history,
            'params':params,
            'best_model':best_model,
            'layers':layers}

def validate(X_val, y_val,params, layers):
    X_val, _ = addBias(X_val,1)
    feed_val = forward(X_val, params ,layers)
    loss_val = (loss_func(y_val, feed_val))
    acc_val = accuracy(y_val,feed_val['y_hat'])
    return loss_val,acc_val

def plot_history(history):
    fig, ax = plt.subplots(ncols = 2, figsize = (12,8))
    ax[0].plot(history['loss'])
    ax[0].plot(history['loss_vall'])
    ax[0].set_title('Loss Graph')
    ax[0].set_xlabel('Epochs')
    ax[0].set_ylabel('Loss')
    
    ax[1].plot(history['acc'])
    ax[1].plot(history['acc_val'])
    ax[1].set_title('Acc Graph')
    ax[1].set_xlabel('Epochs')
    ax[1].set_ylabel('Accuracy')

# Load dataset

In [ ]:
import copy

In [ ]:
#Reading the competition data into a pandas dataframe
train_df = pd.read_csv('../input/titanic/train.csv')
test_df = pd.read_csv('../input/titanic/test.csv')

# Basic feature Engineering
- From a previous notebook

In [ ]:
train_df.columns

In [ ]:
def feature_(data, train = True):
    sub = data[['PassengerId','Name']]
    data = data.drop(['Name', 'PassengerId','Ticket','Cabin','Embarked'], axis = 1)

    data.loc[data['Parch'] > 1, 'Parch'] = 1
    data.loc[data['SibSp'] > 1, 'SibSp'] = 1
    data.loc[data['Sex'] == 'male', 'Sex'] = 1
    data.loc[data['Sex'] == 'female', 'Sex'] = 0
    data.loc[data['Age'].isnull(), 'Age'] =  data['Age'].median()
    
    data['groups'] = data['Sex']
    data['groups'][data['Age'] <= 12] = 3
    
    data = pd.get_dummies(data,columns = ['groups'],dummy_na=False, prefix = 'G_')
    data = data.drop('G__3', axis = 1)
    
   
    data.loc[data['Fare'] > 150, 'Fare'] =  150
    data.loc[data['Fare'].isnull(), 'Fare'] =  data['Fare'].median()
    
    data =  data.drop(['Age','Sex','Parch','SibSp','Age_cat'],axis =1,errors='ignore')
    
   
    scaler = StandardScaler()
    
   
    data_ = data.copy()
    
    if train:
        data_ = data_.drop('Survived',axis =1)
    
    
    cols = data_.columns
    data_ = scaler.fit_transform(data_)
    data_ = pd.DataFrame(data_, columns = cols)
    if train:
        data_['Survived'] = data['Survived']
    return data_,sub


train_df_a, _ = feature_(train_df)
test_df_a, name = feature_(test_df, False)
train_df_a.head()

In [ ]:
X = train_df_a.drop('Survived', axis = 1)
y = train_df_a['Survived']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.4,stratify = y)

X_train = X_train.values.T
y_train = y_train.values.reshape(1,-1)

X_test = X_test.values.T
y_test = y_test.values.reshape(1,-1)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

# Simple Training

In [ ]:
np.random.seed(101)
model = train_model(X_train, X_test, y_train, y_test,10000, 15,lr= 5e-1,lr_factor = 0.95)

In [ ]:
plot_history(model['history'])

# Submission

In [ ]:
from sklearn.metrics import classification_report
X_test_, _ = addBias(test_df_a.values.T,1)
feed = forward(X_test_, model['best_model'],model['layers'])



name['Survived'] = list(map(int, feed['y_hat'][0] > 0.5))
name[['PassengerId','Survived']].to_csv('./submission.csv', index = False)

In [ ]:
name